# Semana 9 — Lista de Prática SQL para Data Warehouse

Este notebook contém **10 exercícios progressivos** para praticar os conteúdos da Semana 9 usando o mesmo modelo de dados trabalhado em aula.

## Conteúdos praticados

- `SELECT`, `WHERE`, `ORDER BY` e `LIMIT`
- `INNER JOIN` entre fato e dimensões
- `LEFT JOIN`
- `GROUP BY`, `SUM`, `COUNT` e `AVG`
- `WHERE` antes da agregação
- `HAVING` depois da agregação
- `DATE_TRUNC` e `EXTRACT`

## Tabelas usadas

- `fato_vendas`: tabela fato com as vendas
- `dim_cliente`: dimensão com cidade e estado do cliente
- `dim_produto`: dimensão com categoria do produto
- `dim_tempo`: dimensão com datas, ano, mês, trimestre e ano/mês

> Orientação: escreva sua resposta dentro da célula de código de cada exercício, substituindo o comentário `-- escreva sua consulta aqui`.


## 1. Preparação do ambiente

Execute as células abaixo antes de iniciar os exercícios. Os arquivos `.csv` devem estar na mesma pasta deste notebook. Caso estejam em outra pasta, altere o valor da variável `BASE_PATH`.


In [4]:
# Instalação, caso esteja usando Google Colab ou ambiente sem DuckDB
# !pip install duckdb pandas

In [ ]:
import pandas as pd
import duckdb
from pathlib import Path

# Altere aqui se os arquivos estiverem em outra pasta
BASE_PATH = Path('F:\Estudos\Senai_dados\Semana_9\Dataset_schema')# CASA
#BASE_PATH = Path(r'C:\Users\victorh.santos\OneDrive - SEB - Sistema Educacional Brasileiro\Estudos\Senai_dados\Semana_9\Dataset_schema')# COC
dim_cliente = pd.read_csv(BASE_PATH / 'dim_cliente.csv')
dim_produto = pd.read_csv(BASE_PATH / 'dim_produto.csv')
dim_tempo = pd.read_csv(BASE_PATH / 'dim_tempo.csv')
fato_vendas = pd.read_csv(BASE_PATH / 'fato_vendas.csv')

banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente)
banco_dados.register('dim_produto', dim_produto)
banco_dados.register('dim_tempo', dim_tempo)
banco_dados.register('fato_vendas', fato_vendas)

print('Tabelas carregadas com sucesso!')

<>:6: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
<>:6: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
C:\Users\victorh.santos\AppData\Local\Temp\ipykernel_14420\3652791018.py:6: SyntaxWarning: "\E" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\E"? A raw string is also an option.
  BASE_PATH = Path('F:\Estudos\Senai_dados\Semana_9\Dataset_schema')# CASA


Tabelas carregadas com sucesso!


## 2. Consulta rápida das tabelas

Use esta parte para lembrar quais colunas existem em cada tabela.


In [9]:
print('fato_vendas')
display(fato_vendas.head())

print('dim_cliente')
display(dim_cliente.head())

print('dim_produto')
display(dim_produto.head())

print('dim_tempo')
display(dim_tempo.head())

fato_vendas


,venda_sk,order_id,order_item_id,cliente_sk,produto_sk,tempo_sk,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,1,65558,25866,64,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,1,34266,27231,125,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,1,34956,22625,222,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,1,51764,15404,3,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,1,7603,8863,512,199.90,18.14,218.04


dim_cliente


,cliente_sk,customer_id,customer_unique_id,customer_city,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


dim_produto


,produto_sk,product_id,product_category_name,product_category_name_english
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,2,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,3,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,4,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,5,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


dim_tempo


,tempo_sk,data,ano,mes,dia,trimestre,ano_mes
0,1,2017-10-02,2017,10,2,4,2017-10
1,2,2018-07-24,2018,7,24,3,2018-07
2,3,2018-08-08,2018,8,8,3,2018-08
3,4,2017-11-18,2017,11,18,4,2017-11
4,5,2018-02-13,2018,2,13,1,2018-02


# 10 exercícios de prática

## Exercício 1 🟢 — Primeiras vendas da tabela fato

Liste as **20 primeiras vendas** da tabela `fato_vendas`.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_produto`
- `valor_frete`
- `valor_total`

**Objetivo:** praticar `SELECT`, `FROM` e `LIMIT`.

In [ ]:
banco_dados.sql("""
SELECT 
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


## Exercício 2 🟢 — Filtro com WHERE

Liste as vendas em que o `valor_total` seja **maior que 300**.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_total`

Ordene o resultado do **maior para o menor valor total**.

**Objetivo:** praticar `WHERE` e `ORDER BY DESC`.

In [ ]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_total
FROM fato_vendas
WHERE valor_total > 300
ORDER BY valor_total DESC
""").df()

,venda_sk,order_id,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4764.34
...,...,...,...
8498,17744,299ad5e33a5f4fe1efcac35b99b0fad8,300.21
8499,105349,f4ae2ab52b8c6e093b06d6656ee4d351,300.12
8500,11827,1b6c00256bc49004b1d059039306c08e,300.12
8501,25863,3c38b53bb7d6f3c9f00d993a80f8cdd5,300.11


## Exercício 3 🟢 — Ranking de fretes

Liste as **15 vendas com maior valor de frete**.

Mostre as colunas:

- `venda_sk`
- `order_id`
- `valor_frete`
- `valor_total`

**Objetivo:** praticar ordenação e criação de ranking com `ORDER BY` e `LIMIT`.

In [ ]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM fato_vendas
ORDER BY valor_total DESC
LIMIT 15
""").df()

,venda_sk,order_id,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,74.34,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,91.78,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,113.45,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,195.76,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,75.27,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,104.51,4163.51


## Exercício 4 🟡 — INNER JOIN com clientes

Mostre as vendas junto com o **estado e a cidade do cliente**.

Use `INNER JOIN` entre `fato_vendas` e `dim_cliente`.

Mostre as colunas:

- `f.venda_sk`
- `f.order_id`
- `c.customer_city`
- `c.customer_state`
- `f.valor_total`

Mostre apenas 20 linhas.

**Objetivo:** praticar relacionamento entre tabela fato e dimensão cliente.

In [ ]:
banco_dados.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    c.customer_city,
    c.customer_state,
    f.valor_total
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = f.cliente_sk
LIMIT 20
""").df()



,venda_sk,order_id,customer_city,customer_state,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,franca,SP,72.19
1,1,00010242fe8c5a6d1ba2dd792cb16214,sao bernardo do campo,SP,72.19
2,1,00010242fe8c5a6d1ba2dd792cb16214,sao paulo,SP,72.19
3,1,00010242fe8c5a6d1ba2dd792cb16214,mogi das cruzes,SP,72.19
4,1,00010242fe8c5a6d1ba2dd792cb16214,campinas,SP,72.19
5,1,00010242fe8c5a6d1ba2dd792cb16214,jaragua do sul,SC,72.19
6,1,00010242fe8c5a6d1ba2dd792cb16214,sao paulo,SP,72.19
7,1,00010242fe8c5a6d1ba2dd792cb16214,timoteo,MG,72.19
8,1,00010242fe8c5a6d1ba2dd792cb16214,curitiba,PR,72.19
9,1,00010242fe8c5a6d1ba2dd792cb16214,belo horizonte,MG,72.19


## Exercício 5 🟡 — INNER JOIN com produto e tempo

Mostre as vendas com **categoria do produto** e **mês/ano da venda**.

Use `INNER JOIN` entre:

- `fato_vendas` e `dim_produto`
- `fato_vendas` e `dim_tempo`

Mostre as colunas:

- `f.order_id`
- `p.product_category_name_english AS categoria`
- `t.ano_mes`
- `f.valor_total`

Mostre apenas 30 linhas.

**Objetivo:** praticar mais de um `JOIN` na mesma consulta.

In [11]:
banco_dados.sql("""
SELECT
    f.order_id,
    p.product_category_name as nome_produto,
    t.ano_mes,
    f.valor_total
FROM fato_vendas f
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
ORDER BY t.ano_mes
LIMIT 30
""").df()

,order_id,nome_produto,ano_mes,valor_total
0,bfbd0f9bdef84302105ad712db648a6c,beleza_saude,2016-09,47.82
1,bfbd0f9bdef84302105ad712db648a6c,beleza_saude,2016-09,47.82
2,bfbd0f9bdef84302105ad712db648a6c,beleza_saude,2016-09,47.82
3,f781dacd75aa73166785a35929546894,market_place,2016-10,51.62
4,f716dffba1232aaef7c899fb8c14db97,perfumaria,2016-10,354.35
5,14b01335f19b97fce1437168032ae388,moveis_decoracao,2016-10,62.30
6,f688669f48063536e082bb32d634cd46,brinquedos,2016-10,111.22
7,02190241f7190a1f3c7e0df95a749c6a,brinquedos,2016-10,260.73
8,f7886facb594e65f0c41cdbb0583b560,brinquedos,2016-10,154.10
9,14b01335f19b97fce1437168032ae388,moveis_decoracao,2016-10,26.36


## Exercício 6 🟡 — Faturamento por estado

Calcule o **faturamento total por estado do cliente**.

Use `fato_vendas` e `dim_cliente`.

A consulta deve retornar:

- `estado`
- `qtd_pedidos`, contando pedidos únicos com `COUNT(DISTINCT f.order_id)`
- `faturamento_total`, usando `SUM(f.valor_total)`
- `ticket_medio`, usando `AVG(f.valor_total)`

Ordene pelo maior faturamento.

**Objetivo:** praticar `GROUP BY`, `SUM`, `COUNT`, `AVG` e `JOIN`.

In [17]:
banco_dados.sql("""
SELECT
    c.customer_state as estado,
    COUNT(f.order_id) as qtd_pedidos,
    SUM(f.valor_total) as faturamento,
    ROUND(AVG(f.valor_total),2) as ticket_medio
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY customer_state
ORDER BY faturamento DESC
""").df()

,estado,qtd_pedidos,faturamento,ticket_medio
0,SP,46448,5769703.15,124.22
1,RJ,14143,2055401.57,145.33
2,MG,12916,1818891.67,140.82
3,RS,6134,861472.79,140.44
4,PR,5649,781708.80,138.38
5,SC,4097,595127.78,145.26
6,BA,3683,591137.81,160.50
7,DF,2355,346123.35,146.97
8,GO,2277,334212.35,146.78
9,ES,2225,317657.93,142.77


## Exercício 7 🟡 — Receita por categoria somente em SP

Calcule a **receita por categoria de produto** considerando apenas clientes do estado de **SP**.

Use `fato_vendas`, `dim_produto` e `dim_cliente`.

A consulta deve retornar:

- `categoria`
- `qtd_pedidos`
- `receita_total`

Ordene pela maior receita.

**Objetivo:** praticar `WHERE` antes do `GROUP BY`.

In [22]:
banco_dados.sql("""
SELECT
    product_category_name as categoria,
    COUNT(f.order_id) as qtd_pedidos,
    SUM(f.valor_total) as receita
FROM fato_vendas f
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
WHERE c.customer_state = 'SP'
GROUP BY p.product_category_name
 ORDER BY receita DESC
""").df()

,categoria,qtd_pedidos,receita
0,cama_mesa_banho,5157,549061.89
1,beleza_saude,4125,510255.38
2,relogios_presentes,2234,448582.76
3,esporte_lazer,3576,428212.45
4,informatica_acessorios,3099,386280.65
...,...,...,...
68,casa_conforto_2,14,624.56
69,pc_gamer,3,595.53
70,flores,11,558.12
71,cds_dvds_musicais,6,409.64


## Exercício 8 🟠 — HAVING com pedidos por estado

Liste apenas os estados com **mais de 500 pedidos únicos**.

A consulta deve retornar:

- `estado`
- `qtd_pedidos`
- `receita_total`

Ordene do maior para o menor número de pedidos.

**Objetivo:** praticar filtro de grupos com `HAVING`.

In [26]:
banco_dados.sql("""
SELECT
    c.customer_state as estado,
    COUNT(DISTINCT f.order_id) as qtd_pedidos_unicos,
    SUM(f.valor_total) as faturamento,
FROM fato_vendas f
JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY customer_state
HAVING qtd_pedidos_unicos > 500
ORDER BY faturamento DESC
""").df()

,estado,qtd_pedidos_unicos,faturamento
0,SP,40501,5769703.15
1,RJ,12350,2055401.57
2,MG,11354,1818891.67
3,RS,5345,861472.79
4,PR,4923,781708.80
5,SC,3546,595127.78
6,BA,3256,591137.81
7,DF,2080,346123.35
8,GO,1957,334212.35
9,ES,1995,317657.93


## Exercício 9 🟠 — Receita mensal com DATE_TRUNC

Calcule o **faturamento por mês** usando `DATE_TRUNC`.

Use `fato_vendas` e `dim_tempo`.

A consulta deve retornar:

- `mes_referencia`, usando `DATE_TRUNC('month', CAST(t.data AS DATE))`
- `qtd_pedidos`
- `receita_total`

Ordene pelo mês em ordem crescente.

**Objetivo:** praticar agregação temporal com `DATE_TRUNC`.

In [34]:
banco_dados.sql("""
SELECT
    DATE_TRUNC('month', CAST(t.data as DATE)) as ano_mês,
    COUNT(f.order_id) as qtd_pedidos_unicos,
    SUM(f.valor_total) as faturamento,
FROM fato_vendas f
JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
GROUP BY DATE_TRUNC('month', CAST(t.data as DATE))
ORDER BY faturamento DESC
""").df()

,ano_mês,qtd_pedidos_unicos,faturamento
0,2017-11-01,8475,1153364.20
1,2018-04-01,7827,1132878.93
2,2018-05-01,7810,1128774.52
3,2018-03-01,8017,1120598.24
4,2018-01-01,8037,1077887.46
5,2018-07-01,6963,1027807.28
6,2018-06-01,7010,1011978.29
7,2018-08-01,7142,985491.64
8,2018-02-01,7518,966168.41
9,2017-12-01,6187,843078.29


## Exercício 10 🔴 — Desafio BI: sazonalidade por trimestre e categoria

Monte uma análise de **receita por trimestre e categoria**.

Use `fato_vendas`, `dim_produto` e `dim_tempo`.

A consulta deve retornar:

- `ano`, usando `EXTRACT(YEAR FROM CAST(t.data AS DATE))`
- `trimestre`, usando `EXTRACT(QUARTER FROM CAST(t.data AS DATE))`
- `categoria`
- `qtd_pedidos`
- `receita_total`
- `ticket_medio`

Mostre apenas grupos com `receita_total` acima de **50000**.

Ordene por:

1. `ano`
2. `trimestre`
3. `receita_total` do maior para o menor

**Objetivo:** combinar `JOIN`, `GROUP BY`, `EXTRACT`, `HAVING` e `ORDER BY`.

In [39]:
banco_dados.sql("""
SELECT
    EXTRACT(YEAR FROM CAST(t.data as DATE)) as ano,
    EXTRACT('quarter' FROM CAST(t.data as DATE)) as trimestre,
    COUNT(f.order_id) as qtd_pedidos_unicos,
    SUM(f.valor_total) as faturamento,
    ROUND(AVG(f.valor_total),2) as ticket_medio
FROM fato_vendas f
JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
GROUP BY EXTRACT(YEAR FROM CAST(t.data as DATE)), EXTRACT('quarter' FROM CAST(t.data as DATE))
ORDER BY ano, trimestre,faturamento DESC
""").df()

,ano,trimestre,qtd_pedidos_unicos,faturamento,ticket_medio
0,2016,3,3,143.46,47.82
1,2016,4,314,46510.28,148.12
2,2017,1,5668,813052.64,143.45
3,2017,2,10062,1447714.17,143.88
4,2017,3,13950,1913208.93,137.15
5,2017,4,19876,2747559.50,138.24
6,2018,1,23572,3164654.11,134.25
7,2018,2,22647,3273631.74,144.55
8,2018,3,14105,2013298.92,142.74


## Checklist final

Antes de entregar, confira:

- Todas as consultas executaram sem erro.
- As consultas com `JOIN` usam as chaves corretas: `cliente_sk`, `produto_sk` e `tempo_sk`.
- As consultas agregadas usam `GROUP BY` para todas as colunas não agregadas.
- Filtros antes da agregação usam `WHERE`.
- Filtros depois da agregação usam `HAVING`.
- Os resultados estão ordenados conforme o enunciado.
